# BlobNet Dataset Generation Playground

Use this notebook to inspect and modify synthetic dataset generation directly. It is intentionally explicit: edit the parameter cells, rerun, and inspect count maps, coordinates, targets, and generated images.

Recommended kernel: the repo `.venv` Python environment.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "blobnet").exists() and (REPO_ROOT.parent / "blobnet").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from blobnet.synthetic import (
    AseStructureProjectionConfig,
    PeriodicLatticeConfig,
    RandomAtomImageConfig,
    GeneratedAtomImageDataset,
    metadata_collate,
    generate_atom_image,
    render_atom_image,
    sample_atom_coordinates,
    generate_and_save_dataset_splits,
)

print(f"repo: {REPO_ROOT}")
print(f"python: {sys.executable}")
print(f"torch: {torch.__version__}")
print(f"mps available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")

## Plotting Helpers

In [ ]:
def image_stats(image_record: dict) -> dict:
    image = np.asarray(image_record["image"], dtype=np.float32)
    target = np.asarray(image_record["target"], dtype=np.float32)
    count_map = image_record.get("count_map")
    stats = {
        "image_shape": image.shape,
        "image_min": float(image.min()),
        "image_max": float(image.max()),
        "image_sum": float(image.sum()),
        "image_nonzero": int((image > 0).sum()),
        "target_max": float(target.max()),
        "atom_count": int(len(image_record.get("coordinates", []))),
    }
    if count_map is not None:
        count_map = np.asarray(count_map)
        stats.update(
            {
                "count_map_sum": float(count_map.sum()),
                "count_map_nonzero": int((count_map > 0).sum()),
                "count_map_max": float(count_map.max()),
                "count_scale": int(image_record.get("count_scale", -1)),
                "total_counts": int(image_record.get("total_counts", -1)),
            }
        )
    return stats


def show_image_record(image_record: dict, title: str = "image", vmax: float | None = None) -> None:
    image = np.asarray(image_record["image"], dtype=np.float32)
    target = np.asarray(image_record["target"], dtype=np.float32)
    coordinates = np.asarray(image_record.get("coordinates", np.zeros((0, 2))), dtype=np.float32)
    count_map = image_record.get("count_map")

    panels = [("image", image, "gray", vmax)]
    if count_map is not None:
        panels.append(("count_map", np.asarray(count_map), "gray", None))
    panels.extend([("target", target, "magma", 1.0), ("overlay", image, "gray", vmax)])

    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4), constrained_layout=True)
    if len(panels) == 1:
        axes = [axes]
    fig.suptitle(title)
    for ax, (name, panel, cmap, panel_vmax) in zip(axes, panels):
        ax.imshow(panel, cmap=cmap, vmin=0, vmax=panel_vmax)
        if name == "overlay" and len(coordinates):
            ax.scatter(coordinates[:, 1], coordinates[:, 0], s=12, facecolors="none", edgecolors="cyan", linewidths=0.7)
        ax.set_title(name)
        ax.axis("off")
    plt.show()


def show_stats(image_record: dict) -> None:
    for key, value in image_stats(image_record).items():
        print(f"{key}: {value}")

## Core Random Atom Image Config

Edit these values first. This is the main random atom-field distribution.

In [ ]:
seed = 130
rng = np.random.default_rng(seed)

atom_image_config = RandomAtomImageConfig(
    image_shape=(256, 256),
    min_atoms=120,
    max_atoms=180,
    min_separation=10.0,
    min_separation_range=(9.0, 12.0),
    sigma_range=(1.8, 3.2),
    background_range=(0.0, 0.30),
    low_frequency_noise_range=(0.08, 0.30),
    read_noise_std_range=(0.0, 0.0),
    total_counts_range=(7500.0, 7500.0),
    blur_sigma_range=(0.3, 1.1),
)

atom_image_config

## One Random Image

In [ ]:
image_record = generate_atom_image(atom_image_config, rng=np.random.default_rng(seed))
show_stats(image_record)
show_image_record(image_record, title="random atom image", vmax=1.0)

## Poisson Scale Audit

This follows the restored visual model: normalize the clean image, draw a same-shape integer `count_map` from `Poisson(normalized_image * count_scale)`, divide by `count_scale`, add read noise, then normalize the image for training/display.

In [ ]:
total_counts_to_test = [0, 1, 3, 10, 25, 100, 1000, 5000]

scene_rng = np.random.default_rng(seed)
sampled_min_separation = float(scene_rng.uniform(*atom_image_config.min_separation_range))
atom_count = int(scene_rng.integers(atom_image_config.min_atoms, atom_image_config.max_atoms + 1))
coordinates = sample_atom_coordinates(
    atom_image_config,
    rng=scene_rng,
    atom_count=atom_count,
    min_separation=sampled_min_separation,
)
intensities = scene_rng.uniform(atom_image_config.intensity_range[0], atom_image_config.intensity_range[1], size=len(coordinates)).astype(np.float32)
sigmas = scene_rng.uniform(atom_image_config.sigma_range[0], atom_image_config.sigma_range[1], size=len(coordinates)).astype(np.float32)

count_images = []
for index, requested_count_scale in enumerate(total_counts_to_test):
    config = replace(
        atom_image_config,
        total_counts_range=(float(requested_count_scale), float(requested_count_scale)),
        read_noise_std_range=(0.0, 0.0),
    )
    count_image = render_atom_image(
        coordinates=coordinates,
        config=config,
        rng=np.random.default_rng(seed + 1000 + index),
        intensities=intensities,
        sigmas=sigmas,
        target_coordinates=coordinates,
        metadata={"requested_count_scale": requested_count_scale},
    )
    count_images.append(count_image)
    stats = image_stats(count_image)
    print(
        f"scale={requested_count_scale:5d}  map_sum={stats['count_map_sum']:7.1f}  "
        f"map_nonzero={stats['count_map_nonzero']:5d}  image_sum={stats['image_sum']:.4f}  "
        f"image_nonzero={stats['image_nonzero']:5d}"
    )


In [ ]:
for count_image in count_images[:6]:
    show_image_record(
        count_image,
        title=f"requested scale = {count_image['requested_count_scale']}",
        vmax=1.0,
    )

## Counts Per Pixel Audit

Set `total_counts_range=None` and use `counts_per_pixel_range` when you want image size to determine the Poisson scale.

In [ ]:
counts_per_pixel = 10.0 / (atom_image_config.image_shape[0] * atom_image_config.image_shape[1])
counts_per_pixel_config = replace(
    atom_image_config,
    total_counts_range=None,
    counts_per_pixel_range=(counts_per_pixel, counts_per_pixel),
    read_noise_std_range=(0.0, 0.0),
)
counts_per_pixel_image = generate_atom_image(
    counts_per_pixel_config,
    rng=np.random.default_rng(seed + 2000),
)
show_stats(counts_per_pixel_image)
show_image_record(counts_per_pixel_image, title=f"counts_per_pixel = {counts_per_pixel:.6g}", vmax=1.0)


## Parameter Sweep

Change any values here to test sensitivity to sigma, blur, atom count, and total counts.

In [ ]:
sweep_total_counts = [10, 100, 1000]
sweep_sigmas = [(0.8, 1.2), (1.8, 3.2), (3.5, 5.0)]

fig, axes = plt.subplots(len(sweep_sigmas), len(sweep_total_counts), figsize=(10, 10), constrained_layout=True)
for row, sigma_range in enumerate(sweep_sigmas):
    for col, requested_count_scale in enumerate(sweep_total_counts):
        config = replace(
            atom_image_config,
            sigma_range=sigma_range,
            total_counts_range=(requested_count_scale, requested_count_scale),
            read_noise_std_range=(0.0, 0.0),
        )
        image_record = generate_atom_image(config, rng=np.random.default_rng(seed + row * 10 + col))
        ax = axes[row, col]
        ax.imshow(image_record["image"], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"sigma={sigma_range}\nscale={requested_count_scale}, map_sum={image_record['count_map'].sum():.0f}")
        ax.axis("off")
plt.show()


## Periodic Lattice Images

In [ ]:
for lattice_type in ["cubic", "hexagonal"]:
    lattice_config = PeriodicLatticeConfig(
        image_shape=(256, 256),
        lattice_type=lattice_type,
        lattice_spacing_range=(12.0, 16.0),
        jitter_std_range=(0.0, 0.2),
        vacancy_fraction_range=(0.0, 0.03),
        sigma_range=(1.8, 3.2),
        read_noise_std_range=(0.0, 0.0),
        total_counts_range=(5000.0, 5000.0),
    )
    lattice_image = generate_atom_image(lattice_config, rng=np.random.default_rng(seed))
    show_stats(lattice_image)
    show_image_record(lattice_image, title=f"{lattice_type} lattice", vmax=1.0)

## ASE Projected Structures

In [ ]:
for structure_name in ["graphene", "ws2", "sto"]:
    structure_config = AseStructureProjectionConfig(
        image_shape=(256, 256),
        structure_name=structure_name,
        sigma_range=(1.8, 3.2),
        read_noise_std_range=(0.0, 0.0),
        total_counts_range=(5000.0, 5000.0),
    )
    try:
        structure_image = generate_atom_image(structure_config, rng=np.random.default_rng(seed))
    except Exception as exc:
        print(f"{structure_name}: skipped ({exc})")
        continue
    show_stats(structure_image)
    show_image_record(structure_image, title=f"ASE projection: {structure_name}", vmax=1.0)

## Dataloaders and Batch Shapes

In [ ]:
from torch.utils.data import DataLoader

train_dataset = GeneratedAtomImageDataset(4, atom_image_config, seed=seed, return_metadata=False)
val_dataset = GeneratedAtomImageDataset(2, atom_image_config, seed=seed + 100_000, return_metadata=False)
test_dataset = GeneratedAtomImageDataset(2, atom_image_config, seed=seed + 200_000, return_metadata=True)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=metadata_collate)

images, targets = next(iter(train_loader))
print("train images", tuple(images.shape), images.dtype)
print("train targets", tuple(targets.shape), targets.dtype)

test_images, test_targets, metadata = next(iter(test_loader))
print("test images", tuple(test_images.shape))
print("metadata keys", metadata[0].keys())

plt.figure(figsize=(4, 4))
plt.imshow(test_images[0, 0], cmap="gray")
plt.title("first test batch image")
plt.axis("off")
plt.show()


## Optional: Save Tiny Dataset Splits

Set `RUN_SAVE = True` if you want to write a small dataset under `/tmp` for inspection.

In [ ]:
RUN_SAVE = False
output_dir = Path("/tmp/blobnet_notebook_dataset")

if RUN_SAVE:
    saved = generate_and_save_dataset_splits(
        output_dir=output_dir,
        train_samples=3,
        val_samples=1,
        test_samples=1,
        config=atom_image_config,
        seed=seed,
    )
    print({split: len(paths) for split, paths in saved.items()})
    print(f"wrote {output_dir}")
else:
    print("Set RUN_SAVE = True to write a tiny dataset split.")